In [2]:
from pyTNG import utils
from pyTNG import data_interface as _data_interface
import os
import pandas as pd
import numpy as np
import illustris_python as il
from pyTNG.spectra import StarSpectrumFactory
from pyTNG import spectra
import matplotlib.pyplot as plt
import time
import astropy.units as u
import scipy.integrate as integ
import pyTNG.utils as utils
from pyTNG.cosmology import TNGcosmo

In [4]:
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:80% !important; }</style>"))

In [17]:
h = TNGcosmo.h

In [3]:
%load_ext autoreload
%autoreload 2

In [3]:
from df_level0 import *

In [21]:
def get_particle_dist(particles, df, index):
    gal_center = np.array([df.loc[index][('SubhaloPos', 0)], df.loc[index][('SubhaloPos', 1)], df.loc[index][('SubhaloPos', 2)]])
    rel_pos = particles['Coordinates']-gal_center
    radius = df.loc[index][('SubhaloHalfmassRadType', 4)]*2
    dist = np.sqrt(np.sum(np.square(rel_pos), axis=1))
    rel_dist = dist/radius
    particles['rel_dist'] = rel_dist
    return

In [37]:
def get_volume_mass(df, sim_path, snap_num, z):
    counter = 0
    volumes = []
    masses = []
    mass_to_g = (1*u.Msun).to(u.kg).value*1e10/h
    volume_to_cm3 = ((1*u.kpc).to(u.cm)/h/(1+z))**3
    for idx in df.index:
        gas = il.snapshot.loadSubhalo(sim_path, snap_num, idx, 'gas')
        get_particle_dist(gas, df, idx)
        gas_in_rad = gas['rel_dist']<1
        gas_masses = gas['Masses'][gas_in_rad]
        gas_densities = gas['Density'][gas_in_rad]
        gas_volumes = gas_masses/gas_densities
        volumes.append(np.sum(gas_volumes))
        masses.append(np.sum(gas_masses))
        
        if counter % 100 == 0:
            print(f'{100*counter/len(df):.2f}% of all halos processed')
        counter += 1
    df[('masses', 0)] = np.array(masses)*mass_to_g
    df[('volumes', 0)] = np.array(volumes)*volume_to_cm3
    return

In [ ]:
def get_column_height_dens(df):
    rad = df[('SubhaloHalfmassRadType',4)]*2
    dist_to_cm = (1*u.kpc).to(u.cm)/h/(1+z)
    rad_cm = rad*dist_to_cm
    areas = rad_cm**2*np.pi
    df[('column_height', 0)] = df[('volumes', 0)]/areas
    df[('column_dens', 0)] = df[('masses', 0)]/df[('column_height', 0)]
    return   

In [16]:
df_path = f'/ptmp/mpa/ivkos/semianalytic_fesc/testing/sn013.pickle'
df = pd.read_pickle(df_path)

In [ ]:
snap_num = 13
df[('SubhaloMassInRadType',0)]
sim, sim_path = get_sim()
z = sim.snap_cat[snap_num].header['Redshift']
get_volume_mass(df, sim_path, snap_num, z)

0.00% of all halos processed
0.04% of all halos processed
0.07% of all halos processed
0.11% of all halos processed
0.14% of all halos processed
0.18% of all halos processed
0.21% of all halos processed
0.25% of all halos processed
0.28% of all halos processed
0.32% of all halos processed
0.36% of all halos processed
0.39% of all halos processed
0.43% of all halos processed
0.46% of all halos processed
0.50% of all halos processed
0.53% of all halos processed
0.57% of all halos processed
0.60% of all halos processed
0.64% of all halos processed
0.68% of all halos processed
0.71% of all halos processed
0.75% of all halos processed
0.78% of all halos processed
0.82% of all halos processed
0.85% of all halos processed
0.89% of all halos processed
0.92% of all halos processed
0.96% of all halos processed
1.00% of all halos processed
1.03% of all halos processed
1.07% of all halos processed
1.10% of all halos processed
1.14% of all halos processed
1.17% of all halos processed
1.21% of all h

In [ ]:
rad = df[('SubhaloHalfmassRadType',4)]*2

In [7]:
df

SubhaloGasMetallicity SubhaloGasMetallicityHalfRad  \
                            0                            0   
0                    0.015532                     0.017928   
1                    0.002725                     0.003610   
2                    0.004793                     0.007448   
3                    0.001956                     0.003759   
4                    0.005315                     0.007526   
...                       ...                          ...   
2368557              0.000578                     0.000597   
2422910              0.000685                     0.000708   
2434511              0.000441                     0.000514   
2521336              0.000406                     0.000402   
2745125              0.000327                     0.000355   

        SubhaloHalfmassRadType SubhaloMassInHalfRad SubhaloMassInHalfRadType  \
                             4                    0                        4   
0                     0.595610             1.791372                 1.380728   
1                     8.928092             0.336186                 0.008784   
2                     6.140252             0.103683                 0.005299   
3                    13.317231             0.193574                 0.002193   
4                     5.533516             0.065053                 0.002343   
...                        ...                  ...                      ...   
2368557               1.293625             0.000540                 0.000004   
2422910               2.607095             0.000968                 0.000003   
2434511               3.430640             0.001582                 0.000003   
2521336               2.234654             0.000920                 0.000003   
2745125               1.533991             0.000673                 0.000002   

        SubhaloMassInRad SubhaloMassInRadType           SubhaloSFRinHalfRad  \
                       0                    0         4                   0   
0               3.268400             0.690410  2.306308          224.748779   
1               0.796984             0.119877  0.012056            1.461907   
2               0.261141             0.028767  0.006582            0.276816   
3               0.471234             0.042249  0.003220            0.078287   
4               0.171911             0.014199  0.003198            0.087283   
...                  ...                  ...       ...                 ...   
2368557         0.001249             0.000103  0.000009            0.000017   
2422910         0.002738             0.000422  0.000010            0.000032   
2434511         0.003374             0.000415  0.000008            0.000016   
2521336         0.002144             0.000232  0.000007            0.000026   
2745125         0.001759             0.000186  0.000005            0.000042   

        SubhaloSFRinRad    SubhaloPos                              
                      0             0             1             2  
0            391.632721  24135.921875  18115.128906  14856.313477  
1              1.798865  24471.667969  18013.396484  14240.319336  
2              0.346269  24264.136719  18214.417969  14564.745117  
3              0.122709  24364.863281  18031.574219  14490.092773  
4              0.115020  24437.298828  17996.375000  14386.402344  
...                 ...           ...           ...           ...  
2368557        0.000017  21072.396484  20731.857422  19424.056641  
2422910        0.000032  25230.380859  18890.257812  30166.308594  
2434511        0.000016  15592.056641   5091.797852  17122.677734  
2521336        0.000026  12621.503906  19735.369141  14953.285156  
2745125        0.000059  20371.802734  28411.167969  10255.405273  

[281349 rows x 13 columns]